# DS-02 – Prévision hebdomadaire de la demande par univers et entrepôt
TuniShop – dossier TS-DATA-DSC-001 – v1.0 – 14/10/2026

**Question** : prévoir à 4 semaines la quantité vendue par univers et par entrepôt (BM-15, PR-1) pour anticiper les réapprovisionnements et les temps forts (BM-20).

**Protocole** : 27 séries hebdomadaires (9 univers × 3 entrepôts) ; test sur les 13 semaines du 29/09/2025 au 28/12/2025 (dont le Black Friday) ; toutes les variables sont décalées d'au moins 4 semaines (horizon de prévision).

In [1]:
import os, json, warnings
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
# Dossier donnees_reference du jeu TuniShop (variable d'environnement ou chemin relatif)
R = os.environ.get("TUNISHOP_REF_DIR", "../donnees_reference") + "/"
FIG = os.environ.get("TUNISHOP_FIG_DIR", "figures") + "/"; os.makedirs(FIG, exist_ok=True)
SEED = 42
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
from sklearn.ensemble import HistGradientBoostingRegressor

## 1. Séries hebdomadaires et calendrier commercial

In [2]:
c=pd.read_csv(R+'commandes.csv',parse_dates=['date_commande']); lc=pd.read_csv(R+'lignes_commande.csv'); p=pd.read_csv(R+'produits.csv'); k=pd.read_csv(R+'categories.csv'); cal=pd.read_csv(R+'dim_calendrier.csv',parse_dates=['date'])
c=c[c.statut_commande!='paiement_echoue']
x=lc.merge(c[['commande_id','date_commande','entrepot_id']]).merge(p[['produit_id','categorie_id']]).merge(k[['categorie_id','categorie']])
x=x[x.entrepot_id.notna()]
x['semaine']=x.date_commande.dt.to_period('W-SUN').dt.start_time
s=x.groupby(['categorie','entrepot_id','semaine']).quantite.sum().reset_index()
idx=pd.MultiIndex.from_product([s.categorie.unique(),s.entrepot_id.unique(),pd.date_range(s.semaine.min(),s.semaine.max(),freq='W-MON')],names=['categorie','entrepot_id','semaine'])
s=s.set_index(['categorie','entrepot_id','semaine']).reindex(idx,fill_value=0).reset_index()
# calendrier hebdo
cal['semaine']=cal.date.dt.to_period('W-SUN').dt.start_time
cw=cal.groupby('semaine').agg(bf=('periode_commerciale',lambda v:(v=='black_friday').sum()),aid=('periode_commerciale',lambda v:v.isin(['pre_aid_fitr','pre_aid_idha']).sum()),
   ram=('est_ramadan','sum'),soldes=('periode_commerciale',lambda v:v.str.startswith('soldes').sum()),rentree=('periode_commerciale',lambda v:(v=='rentree_scolaire').sum()),fer=('est_jour_ferie','sum')).reset_index()
s=s.merge(cw,on='semaine',how='left').sort_values(['categorie','entrepot_id','semaine'])
G=s.groupby(['categorie','entrepot_id']).quantite
for L in [4,5,6,8,52]: s[f'lag{L}']=G.shift(L)
s['mm4_l4']=G.transform(lambda v:v.shift(4).rolling(4).mean())
s['mm13_l4']=G.transform(lambda v:v.shift(4).rolling(13).mean())
s['semaine_annee']=s.semaine.dt.isocalendar().week.astype(int); s['annee']=s.semaine.dt.year
s['cat']=s.categorie.astype('category').cat.codes; s['ent']=s.entrepot_id.astype('category').cat.codes


## 2. Découpage, modèle et méthodes de référence
- **Moyenne mobile 4 semaines** (référence PR-1) : moyenne des 4 dernières semaines connues à l'horizon.
- **Saisonnière naïve** : même semaine de l'année précédente × croissance des 26 dernières semaines connues.
- **Gradient boosting** (perte de Poisson) : un modèle global pour les 27 séries, avec retards, moyennes mobiles et calendrier commercial.

In [3]:
debut_test=pd.Timestamp('2025-09-29'); fin=pd.Timestamp('2025-12-29')
te=s[(s.semaine>=debut_test)&(s.semaine<fin)].copy(); tr=s[(s.semaine<debut_test-pd.Timedelta(weeks=3))&s.lag52.notna()].copy()
F=['lag4','lag5','lag6','lag8','lag52','mm4_l4','mm13_l4','bf','aid','ram','soldes','rentree','fer','semaine_annee','cat','ent']
m=HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,random_state=42,loss='poisson',categorical_features=[F.index('cat'),F.index('ent')]); m.fit(tr[F],tr.quantite)
te['gb']=m.predict(te[F]).clip(0)
te['mm4']=te.mm4_l4  # moyenne mobile des 4 semaines connues à l'horizon 4
a=s[(s.semaine<debut_test)&(s.semaine>=debut_test-pd.Timedelta(weeks=26))].quantite.sum(); b=s[(s.semaine<debut_test-pd.Timedelta(weeks=52))&(s.semaine>=debut_test-pd.Timedelta(weeks=78))].quantite.sum()
croissance=a/b; print('croissance',round(croissance,3)); te['snaive']=te.lag52*croissance


croissance 1.339


## 3. Évaluation

In [4]:
def wape(y,f): return np.abs(y-f).sum()/y.sum()
lignes=[]
for n,lib in [('mm4','Moyenne mobile 4 semaines (référence)'),('snaive','Saisonnière naïve'),('gb','Gradient boosting')]:
    bf=te[te.bf>0]
    lignes.append(dict(methode=lib,MAE=np.abs(te.quantite-te[n]).mean(),WAPE=wape(te.quantite,te[n]),WAPE_semaine_black_friday=wape(bf.quantite,bf[n])))
resultats=pd.DataFrame(lignes).round(3); resultats

,methode,MAE,WAPE,WAPE_semaine_black_friday
0,Moyenne mobile 4 semaines (référence),26.998,0.455,0.455
1,Saisonnière naïve,18.509,0.312,0.257
2,Gradient boosting,17.864,0.301,0.276


In [5]:
parU=te.groupby('categorie').apply(lambda g: pd.Series({'WAPE_reference':wape(g.quantite,g.mm4),'WAPE_modele':wape(g.quantite,g.gb),'quantite':g.quantite.sum()})).round(3).sort_values('quantite',ascending=False); parU

,WAPE_reference,WAPE_modele,quantite
categorie,,,
Fournitures & Livres,0.824,0.308,4115.0
Électronique & High-Tech,0.435,0.356,3805.0
Mode,0.357,0.278,3277.0
Beauté & Santé,0.338,0.266,2831.0
Maison & Cuisine,0.273,0.236,2799.0
Épicerie fine,0.323,0.343,1347.0
Bébé & Enfants,0.430,0.297,1309.0
Électroménager,0.429,0.361,803.0
Sport & Loisirs,0.371,0.343,540.0


In [6]:
tot=te.groupby('semaine')[['quantite','mm4','snaive','gb']].sum()
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(tot.index,tot.quantite,'k-o',label='Réel',lw=2)
for n,lib in [('mm4','Moyenne mobile 4 sem.'),('snaive','Saisonnière naïve'),('gb','Gradient boosting')]: ax.plot(tot.index,tot[n],'--',label=lib)
ax.set_title('Quantités hebdomadaires totales – période de test'); ax.set_ylabel('Articles'); ax.legend(fontsize=8); fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIG+'ds02_prevision.png',dpi=150)

## 4. Conclusion

In [7]:
r=resultats.set_index('methode'); succes=bool(r.loc['Gradient boosting','MAE']<r.loc['Moyenne mobile 4 semaines (référence)','MAE'])
json.dump(dict(resultats=resultats.to_dict('records'),par_univers=parU.reset_index().to_dict('records'),succes=succes,croissance=float(croissance),n_series=27,semaines_test=int(te.semaine.nunique())),open(FIG+'ds02_resultats.json','w'),ensure_ascii=False,default=float)
print('Critère PR-1 (MAE modèle < MAE moyenne mobile) atteint :',succes)

Critère PR-1 (MAE modèle < MAE moyenne mobile) atteint : True
